# Predict heterologous pathways

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you should be able to:

1. Explain why a host needs heterologous (foreign) reactions to make many target chemicals.
2. Describe how cameo combines a host model with a universal reaction database to search for pathways.
3. Run a pathway prediction, read the predicted reactions, and evaluate each pathway with a production envelope.
4. Judge how the choice of host and reaction database changes the predictions.

**Estimated time:** 60 min (pathway predictions take a few minutes each with the free GLPK solver).

**Prerequisites:** loading and optimising models (notebooks 01-02), production envelopes (notebook 04).

### Background

Most interesting products, such as flavours, drugs or bioplastic monomers, cannot be made by the native metabolism of common hosts. Metabolic engineers therefore insert **heterologous pathways**: genes from other organisms that encode the missing enzymes. Because the number of known biochemical reactions is very large, it helps to let a computer propose candidate routes from the host's metabolites to the product.

Cameo ([Cardoso et al., 2018](https://doi.org/10.1021/acssynbio.7b00423)) does this with a **universal model**: a large collection of known reactions, taken from the MetaNetX resource ([Moretti et al., 2021](https://doi.org/10.1093/nar/gkaa992)), which reconciles reaction and metabolite identifiers across databases such as BiGG, Rhea, KEGG and BRENDA. The predictor

1. merges the host model with the universal model, connecting shared metabolites through *adapter* reactions (MetaNetX identifiers are used to match host and universal metabolites),
2. gives every universal reaction an on/off binary switch, and
3. solves a mixed-integer linear program (MILP) that finds the **smallest number of heterologous reactions** that allows a minimum production flux of the product.

After each solution, an *integer cut* forbids exactly that set of reactions, and the MILP is solved again to find the next-shortest pathway. This yields a ranked list of alternative pathways.

In this notebook we use the yeast *Saccharomyces cerevisiae* model **iMM904** as host and **vanillin**, the main flavour compound of vanilla, as target. Yeast is a robust, food-safe production host, and microbial vanillin production is an established example of metabolic engineering. Yeast has no native vanillin pathway, so the predictor must find heterologous routes, starting from intermediates of the shikimate pathway (the pathway that makes aromatic amino acids).

Keep in mind that the predictions only use stoichiometry: they do not check whether the enzymes exist in a form that is active in yeast, whether intermediates are toxic, or how much protein the pathway costs.

In [ ]:
import re
from cobra.io import read_sbml_model
import cameo_compat  # noqa: F401 (patches cameo for modern Python/cobra)
from cameo.strain_design import pathway_prediction
from cameo.visualization.plotting.with_plotly import PlotlyPlotter
plotter = PlotlyPlotter()

Load the yeast genome-scale model iMM904.

In [ ]:
model = read_sbml_model('data/iMM904.xml.gz')

Create a `PathwayPredictor` for the host. Without further arguments it uses the default universal model `metanetx_universal_model_bigg` (reactions from BiGG models, mapped through MetaNetX). The universal models ship with cameo, so no download is needed. Building the combined model takes about a minute.

In [ ]:
predictor = pathway_prediction.PathwayPredictor(model)

Search for up to four pathways to vanillin. Each prediction is one MILP solve followed by a check that the pathway really allows production; with GLPK this takes a few minutes in total.

In [ ]:
pathways = predictor.run(product="vanillin", max_predictions=4)

The output lists each pathway as a set of heterologous reactions, identified by MetaNetX reaction IDs (MNXR...). To plug a pathway into the host, cameo also uses adapter reactions (linking host and universal metabolite identifiers) and an exchange reaction for the product; these are bookkeeping, not enzymes. Let's look at the first pathway as a table with the full reaction equations.

In [ ]:
pathways.pathways[0].data_frame

In our run the first (shortest) pathway has three reactions: 3-dehydroshikimate, a native shikimate pathway intermediate, is dehydrated to 3,4-dihydroxybenzoate (protocatechuate), which is methylated to vanillate and finally reduced to vanillin. Look closely at the methylation step: it is the vanillate monooxygenase (O-demethylase) reaction run *backwards*, which is feasible here only because the universal model treats every reaction as reversible (bounds -1000 to 1000). In real vanillin yeast strains the methyl group is added by an O-methyltransferase instead. The first yeast strains making vanillin from glucose used a 3-dehydroshikimate dehydratase, an aromatic carboxylic acid reductase and an O-methyltransferase ([Hansen et al., 2009](https://doi.org/10.1128/aem.02681-08)). Predictions are hypotheses: check every step for direction, cofactors and enzyme availability.

Finally, we plug each pathway into the host and compute its **production envelope** (vanillin flux versus growth). Compare the pathways: a higher maximum vanillin flux at zero growth means a better theoretical yield, and the slope shows how strongly production competes with growth.

In [ ]:
pathways.plot_production_envelopes(plotter, model, objective=model.reactions.BIOMASS_SC5_notrace)

## Exercise (30 min)

* Predict pathways for a product of your choice in a different host organism. For a quick run, try **propane-1,3-diol** (a polymer building block) in *E. coli* (`data/iJO1366.xml.gz`), with `max_predictions=2`.
* Do predictions change if you use a different universal reaction database? The available databases are attributes of `cameo.models.universal`.

### Solution

First, list the universal models shipped with cameo. Larger databases (with Rhea, KEGG and BRENDA reactions) contain more reactions, which can reveal additional routes but make each MILP considerably slower.

In [ ]:
from cameo.models import universal

[name for name in dir(universal) if name.startswith("metanetx")]

Predict pathways to propane-1,3-diol in *E. coli* iJO1366 with the default universal model (about 2 minutes).

In [ ]:
ecoli = read_sbml_model("data/iJO1366.xml.gz")
predictor_pdo = pathway_prediction.PathwayPredictor(ecoli)
pathways_pdo = predictor_pdo.run(product="propane-1,3-diol", max_predictions=2)
pathways_pdo.pathways[0].data_frame

In [ ]:
pathways_pdo.plot_production_envelopes(plotter, ecoli, objective=ecoli.reactions.BIOMASS_Ec_iJO1366_core_53p95M)

Only one pathway is found with the default database: propane-1,3-diol is made from glycerol in two steps (glycerol dehydratase to 3-hydroxypropanal, then NADH-dependent reduction to propane-1,3-diol), the route used in industrial PDO strains. The search stops after one pathway because no second pathway exists in this database.

To compare databases, pass `universal_model=universal.metanetx_universal_model_bigg_rhea` to `PathwayPredictor` (code below, commented out because it takes about 10 minutes with GLPK). When we ran it, the larger database returned **two** pathways: the same NADH-dependent route, plus a variant in which 3-hydroxypropanal is reduced with **NADPH** (Rhea reaction 35599). A bigger database therefore adds cofactor alternatives, which matter for engineering because the cell balances NADH and NADPH differently. The even larger KEGG and BRENDA databases did not finish within 20 minutes for vanillin in yeast with GLPK.

In [ ]:
# Takes about 10 minutes with GLPK:
# predictor_pdo_rhea = pathway_prediction.PathwayPredictor(ecoli, universal_model=universal.metanetx_universal_model_bigg_rhea)
# pathways_pdo_rhea = predictor_pdo_rhea.run(product="propane-1,3-diol", max_predictions=2)
# for i, pathway in enumerate(pathways_pdo_rhea.pathways):
#     print(f"Pathway {i + 1}:", [(r.id, r.build_reaction_string(True)) for r in pathway.reactions])

## References

- Cardoso, Jensen, Lieven et al. (2018). Cameo: A Python Library for Computer Aided Metabolic Engineering and Optimization of Cell Factories. *ACS Synthetic Biology* 7:1163–1166. [doi.org/10.1021/acssynbio.7b00423](https://doi.org/10.1021/acssynbio.7b00423)
- Hansen, Møller, Kock et al. (2009). De Novo Biosynthesis of Vanillin in Fission Yeast (*Schizosaccharomyces pombe*) and Baker’s Yeast (*Saccharomyces cerevisiae*). *Applied and Environmental Microbiology* 75:2765–2774. [doi.org/10.1128/aem.02681-08](https://doi.org/10.1128/aem.02681-08)
- Moretti, Tran, Mehl et al. (2021). MetaNetX/MNXref: unified namespace for metabolites and biochemical reactions in the context of metabolic models. *Nucleic Acids Research* 49:D570–D574. [doi.org/10.1093/nar/gkaa992](https://doi.org/10.1093/nar/gkaa992)